# Day 1 — GX Architecture and Data Context

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/great-expectations-certified/notebooks/day-01-gx-architecture.ipynb)

**Course:** Great Expectations for Data Quality  
**Day:** 1 of 5  
**Badge:** Learn

---

## What you will learn

By the end of this notebook you will be able to:

- Describe the four core GX objects and how they relate to each other
- Distinguish between `FileSystemDataContext` and `EphemeralDataContext`
- Instantiate a GX context and inspect its configuration
- Map GX concepts onto a real data pipeline

## 0 — Install and version check

Great Expectations is distributed on PyPI. Install it with `pip` and confirm the version before doing anything else — version mismatches are the single most common source of confusion in GX projects.

In [ ]:
# Install great-expectations (skip output with -q to keep the notebook tidy)
!pip install great-expectations -q

In [ ]:
# Confirm the installed version
!great_expectations --version

In [ ]:
import great_expectations as gx

print('great_expectations version:', gx.__version__)

## 1 — GX Architecture Overview

Before writing a single line of validation code it is worth understanding the four main objects that make up a GX project. Think of them as nested layers:

```
DataContext
  └── Datasource
        └── DataAsset  →  BatchRequest  →  Batch
              └── Validator
                    └── ExpectationSuite
```

| Object | Responsibility |
|---|---|
| **DataContext** | Top-level object. Owns all configuration, stores, and plugins. Entry point for every GX operation. |
| **Datasource** | Knows how to connect to a data backend (Pandas, Spark, SQLite, Snowflake, …). |
| **DataAsset** | A named, logical pointer to a table, file, or query within a Datasource. |
| **Validator** | Loads a concrete Batch from a DataAsset and runs Expectations against it. |
| **ExpectationSuite** | A named, serialisable collection of Expectations. Stored in the context for reuse. |

> **Architecture reference:** [GX OSS Overview](https://docs.greatexpectations.io/docs/reference/learn/conceptual_guides/gx_overview)

## 2 — FileSystemDataContext vs EphemeralDataContext

GX offers two primary context modes:

| Mode | How to create | Persists config? | Best for |
|---|---|---|---|
| `EphemeralDataContext` | `gx.get_context()` | No — lives in memory only | Notebooks, CI checks, one-off scripts |
| `FileSystemDataContext` | `gx.get_context(mode='file')` | Yes — writes `great_expectations/` folder | Production pipelines, team repos |
| `CloudDataContext` | `gx.get_context(mode='cloud')` | Yes — GX Cloud SaaS | Enterprise, collaborative workflows |

In this notebook we use `EphemeralDataContext` so the demos run cleanly in Colab without touching the filesystem.

> **Reference:** [Data Context conceptual guide](https://docs.greatexpectations.io/docs/reference/learn/conceptual_guides/data_context)

In [ ]:
# Instantiate an EphemeralDataContext — no filesystem writes
context = gx.get_context()

print('Context type:', type(context).__name__)
print('Context config version:', context.config.config_version)

## 3 — Inspecting the DataContext

Even an `EphemeralDataContext` has a fully-structured configuration object. Let us inspect its top-level keys to understand what GX tracks for us.

In [ ]:
import json

# Serialise the context config to a readable dict
config_dict = context.config.to_json_dict()

# Print the top-level keys
print('Top-level config keys:')
for key in config_dict:
    print(' ', key)

In [ ]:
# Inspect the stores section — GX uses stores to persist suites, results, and docs
stores = config_dict.get('stores', {})
print('Configured stores:')
for store_name in stores:
    print(' ', store_name)

### What are stores?

GX uses **stores** to persist different kinds of objects:

- `expectations_store` — serialised `ExpectationSuite` objects (JSON files or DB rows)
- `validations_store` — `ValidationResult` objects from each `validator.validate()` call
- `evaluation_parameter_store` — dynamic parameters injected at validation time
- `checkpoint_store` — reusable `Checkpoint` definitions
- `data_docs_sites` — auto-generated HTML documentation

In `EphemeralDataContext` all stores live in memory. In `FileSystemDataContext` they are written to the `great_expectations/` directory.

## 4 — The Four Main Objects: Hands-on Mapping

Let us build each object one step at a time and print its type so the hierarchy becomes concrete rather than abstract.

In [ ]:
import pandas as pd

# --- Step 1: DataContext (already created above as `context`) ---
print('1. DataContext')
print('   type:', type(context).__name__)
print()

# --- Step 2: Datasource ---
# We add a Pandas datasource backed by an in-memory dict of DataFrames
datasource = context.data_sources.add_pandas(name='in_memory_pandas')
print('2. Datasource')
print('   type:', type(datasource).__name__)
print('   name:', datasource.name)
print()

In [ ]:
# --- Step 3: DataAsset and Batch ---
# Generate a synthetic DataFrame to validate
import pandas as pd
import numpy as np

np.random.seed(42)
sample_df = pd.DataFrame({
    'user_id':   range(1, 101),
    'age':       np.random.randint(18, 80, 100),
    'email':     [f'user{i}@example.com' for i in range(1, 101)],
    'revenue':   np.round(np.random.uniform(10.0, 500.0, 100), 2),
})

print('Sample DataFrame shape:', sample_df.shape)
print(sample_df.head(3))

In [ ]:
# Add a DataAsset — a named logical pointer to this DataFrame
data_asset = datasource.add_dataframe_asset(name='user_revenue')

print('3. DataAsset')
print('   type:', type(data_asset).__name__)
print('   name:', data_asset.name)

# Build a BatchRequest and load a Batch
batch_request = data_asset.build_batch_request(dataframe=sample_df)
print()
print('   BatchRequest datasource_name:', batch_request.datasource_name)
print('   BatchRequest data_asset_name:', batch_request.data_asset_name)

In [ ]:
# --- Step 4: ExpectationSuite and Validator ---
suite = context.suites.add(gx.ExpectationSuite(name='day1_suite'))

print('4a. ExpectationSuite')
print('   type:', type(suite).__name__)
print('   name:', suite.name)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite=suite,
)

print()
print('4b. Validator')
print('   type:', type(validator).__name__)
print('   columns:', list(validator.head().columns))

## 5 — Object Relationship Diagram

Here is a simple ASCII diagram of the object hierarchy we just built:

```
EphemeralDataContext  (context)
  │
  ├── PandasDatasource  (datasource)  ← knows how to read DataFrames
  │     └── DataFrameAsset  (data_asset)  ← named pointer: 'user_revenue'
  │           └── BatchRequest  →  Batch  (a concrete slice at runtime)
  │
  ├── ExpectationSuite  (suite)  ← 'day1_suite', reusable ruleset
  │
  └── Validator  ← Batch + Suite combined; runs expectations
```

Key insight: the **DataContext** is the single source of truth. You never instantiate Datasources or Suites directly — you always go through `context.data_sources.add_*()` and `context.suites.add()` so that GX can manage persistence and references.

## 6 — FileSystemDataContext: What Gets Scaffolded

When you run `gx.get_context(mode='file')` in a project directory, GX writes this folder structure:

```
great_expectations/
├── great_expectations.yml      # master config: datasources, stores, data docs
├── expectations/               # one JSON file per ExpectationSuite
│   └── my_suite.json
├── checkpoints/                # reusable validation pipelines
│   └── my_checkpoint.yml
├── uncommitted/
│   ├── data_docs/              # auto-generated HTML validation reports
│   └── validations/            # ValidationResult JSON (often git-ignored)
└── plugins/
    └── custom_data_docs/
```

> **Tip:** Commit the `great_expectations/` folder (excluding `uncommitted/`) to version control. This makes your validation layer reproducible across machines — team members `git clone` and get the exact same suite definitions and checkpoint configs.

## Challenge — Extend the Object Map

Complete the tasks below in the cells provided:

1. Add a **second** Pandas datasource named `'orders_source'`
2. Create a synthetic `orders_df` with columns `order_id`, `user_id`, `amount`, `status`
3. Add a `DataFrameAsset` named `'orders'` to the new datasource
4. Build a `BatchRequest` and print its `datasource_name` and `data_asset_name`
5. Create a new `ExpectationSuite` named `'orders_suite'` and a `Validator` for it
6. Print the validator column list

In [ ]:
# Your solution here
# Step 1: add orders datasource

# Step 2: create synthetic orders_df

# Step 3: add DataFrameAsset

# Step 4: build BatchRequest and print names

# Step 5: create ExpectationSuite and Validator

# Step 6: print validator columns


## Day 1 Recap

| Concept | Key takeaway |
|---|---|
| GX Architecture | Four nested objects: DataContext → Datasource → DataAsset/Batch → Validator+Suite |
| EphemeralDataContext | In-memory, no filesystem writes, ideal for notebooks and CI |
| FileSystemDataContext | Writes `great_expectations/` folder; commit it for reproducibility |
| Stores | GX internal persistence layer for suites, results, checkpoints, and docs |
| Entry point | Always use `gx.get_context()` — never instantiate context subclasses directly |

---

**Up next — Day 2:** Connecting to real data sources: Pandas filesystem and SQLite datasources, BatchRequest parameters, and loading your first real batch.